In [1]:
import requests
import torch
from transformers import BertTokenizer, BertForPreTraining

In [2]:
# initialize the tokenizer, model and get data
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForPreTraining.from_pretrained('bert-base-uncased')
data = requests.get("https://raw.githubusercontent.com/jamescalam/transformers/main/data/text/meditations/clean.txt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

In [3]:
# extract text from data retrived and tokenize it
text = data.text
text = text.split("\n")
text[:3]

['From my grandfather Verus I learned good morals and the government of my temper.',
 'From the reputation and remembrance of my father, modesty and a manly character.',
 'From my mother, piety and beneficence, and abstinence, not only from evil deeds, but even from evil thoughts; and further, simplicity in my way of living, far removed from the habits of the rich.']

In [4]:
# First We are going to prepare the dataset for the NSP training
# we can only use the paragraphs with multiple sentences in them, only then we can one sentence follows the other. So we are exclusing the single sentence paragraphs
bag = [sentence for para in text for sentence in para.split('.') if sentence!='']
bag_size = len(bag)

In [6]:
#Now in the NSap trainign samples about 50% of the samples are coorect sentences following the sentence and the rest of the sentences are random pairs
# label : 1 => sentence is not the next sentence
# label : 0 => sentence is the next sentence

import random

sentence_a = []
sentence_b = []
label = []

# iterate through the text and pair 50% of the sentences randomly and rest of them in order
for paragraphs in text:
  sentences = [
      sentence for sentence in paragraphs.split('.') if sentence!=''
  ]
  num_sentences = len(sentences)
  # consider only multiple sentence paragraphs
  if num_sentences > 1 :
    start = random.randint(0, num_sentences-2) # we need to able to pair the sentences so we only interate to the last but 1 sentence
    sentence_a.append(sentences[start])
    if random.random() > 0.5 : # randomly assign some sentence
      sentence_b.append(bag[random.randint(0, bag_size-1)])
      label.append(1) # since its random we assign label 1
    else:
      sentence_b.append(sentences[start+1])
      label.append(0)

In [7]:
# now the samples are ready , tokenize the samples
inputs = tokenizer(sentence_a, sentence_b, return_tensors="pt", max_length = 512, padding = True, truncation = True)
inputs

{'input_ids': tensor([[  101,  2002,  2001,  ...,     0,     0,     0],
        [  101,  2010,  7800,  ...,     0,     0,     0],
        [  101,  2000,  1996,  ...,     0,     0,     0],
        ...,
        [  101,  3459,  2185,  ...,     0,     0,     0],
        [  101,  2043, 15223,  ...,     0,     0,     0],
        [  101,  7887,  3288,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [11]:
# now we need to add our labels tht inputs
# format labels so they fit to the shape of inputs
inputs['next_sentence_label'] = torch.LongTensor([label]).T

In [12]:
inputs.keys()

KeysView({'input_ids': tensor([[  101,  2002,  2001,  ...,     0,     0,     0],
        [  101,  2010,  7800,  ...,     0,     0,     0],
        [  101,  2000,  1996,  ...,     0,     0,     0],
        ...,
        [  101,  3459,  2185,  ...,     0,     0,     0],
        [  101,  2043, 15223,  ...,     0,     0,     0],
        [  101,  7887,  3288,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'next_sentence_label': tensor([[1],
        [1],
        [0],
        [0],
        [1],
        [1],
        [1],
        [1],
        [0],


In [13]:
# lets check if  all the input tensors are of similar size
for key, tensor in inputs.items():
  print(key, tensor.shape)

input_ids torch.Size([317, 280])
token_type_ids torch.Size([317, 280])
attention_mask torch.Size([317, 280])
next_sentence_label torch.Size([317, 1])


In [15]:
# now lets create sample data for MLM training
# for the labels we clone the input ids , once we get them we pick some of the input_ids and mask them randomly
inputs['labels'] = inputs.input_ids.detach().clone()

In [16]:
inputs.keys()

KeysView({'input_ids': tensor([[  101,  2002,  2001,  ...,     0,     0,     0],
        [  101,  2010,  7800,  ...,     0,     0,     0],
        [  101,  2000,  1996,  ...,     0,     0,     0],
        ...,
        [  101,  3459,  2185,  ...,     0,     0,     0],
        [  101,  2043, 15223,  ...,     0,     0,     0],
        [  101,  7887,  3288,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'next_sentence_label': tensor([[1],
        [1],
        [0],
        [0],
        [1],
        [1],
        [1],
        [1],
        [0],


In [20]:
# now we create a mask array randomly so we can mask 15% of the inputs
rand = torch.rand(inputs.input_ids.shape)
mask_arr = (rand < 0.15) * (inputs.input_ids != 101) * (inputs.input_ids != 102) * (inputs.input_ids != 0)

# now get the indices where the mask_arr has true value. We set the correspoding input_ids to 103 i.e [MASK]
selection = []

for i in range(inputs.input_ids.shape[0]):
    selection.append(
        torch.flatten(mask_arr[i].nonzero()).tolist()
    )

In [21]:
# now we iterate through the input_ids and set the masks
for i in range(inputs.input_ids.shape[0]):
    inputs.input_ids[i,selection[i]] = 103

In [22]:
# we need to make the dataset
class MeditationsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}
    def __len__(self):
        return len(self.encodings.input_ids)

In [23]:
dataset = MeditationsDataset(inputs)

In [24]:
# get loader
loader = torch.utils.data.DataLoader(dataset, batch_size = 16, shuffle = True)

In [25]:
# check if we have a cuda device and move the model and loader to teh device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [26]:
model.to(device)

BertForPreTraining(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [27]:
model.train()

BertForPreTraining(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [28]:
# we need AdamW optimizer
from torch.optim import AdamW

optim = AdamW(model.parameters(), lr = 5e-5)

In [30]:
from torch.nn import attention
# to track the training we are using tqdm
from tqdm import tqdm

epochs = 2

for epoch in range(epochs):
  loop = tqdm(loader, leave = True)
  for batch in loop:
    optim.zero_grad()
    input_ids = batch['input_ids'].to(device)
    token_type_ids = batch['token_type_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    next_sentence_label = batch['next_sentence_label'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_ids, token_type_ids = token_type_ids, attention_mask = attention_mask, next_sentence_label = next_sentence_label, labels = labels)
    loss = outputs.loss

    loss.backward()
    optim.step()

    loop.set_description(f'Epoch {epoch}')
    loop.set_postfix(loss = loss.item())

  0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_1484/2782019267.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}
Epoch 1: 100%|██████████| 20/20 [00:20<00:00,  1.02s/it, loss=0.638]


In [38]:
# lets try to test it
text = "This is a great [MASK]."
inputs = tokenizer(text, return_tensors = "pt")
inputs = inputs.to(device)
outputs = model(**inputs)
token_logits = outputs.prediction_logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

'>>> This is a great [PAD].'
'>>> This is a great discovery.'
'>>> This is a great thing.'
'>>> This is a great compromise.'
'>>> This is a great change.'
